# Stock data analysis

In [1]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()

In [4]:
from urllib.parse import urlencode
import datetime

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [5]:
class YahooFinanceWebsite:
    def __init__(self, stock_symbol):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.stock_symbol = stock_symbol.upper()

    def __build_url(self, params):
        base_url = f"https://finance.yahoo.com/quote/{self.stock_symbol}/history/"
        query_string = urlencode(params)
        return f"{base_url}?{query_string}"

    def get_stock_data(self):
        datetime_now = datetime.datetime.now()
        datetime_year_ago = datetime_now - datetime.timedelta(days=365)
        params = {"frequency": "1wk", "period1": datetime_year_ago.timestamp(), "period2": datetime_now.timestamp()}
        url = self.__build_url(params)
        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.content, 'html.parser')
        
        title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        html_table_data = soup.find("table")

        return title, html_table_data

In [6]:
def build_stock_analysis_prompt(stock_symbol, title, stock_table_data):
    sys_prompt = r"""You are an assistant that analyzes the contents of HTML formated table that contains data on a specific stock.
    The HTML table contains the date, open price, close price, low and highs aggregated for every week over one year timeframe.
    Ignoring text, tags or html attributes that might be navigation related. 
    Respond in Markdown format"""
    
    user_prompt = f"The data provided below in the HTML table format for {stock_symbol} from the Yahoo Finances.\
    Make the explaination easy enough for a newbie to understand. \
    Analyze and Summarize the trends on this stock:\n{stock_table_data}\n\n\
    Also, calculate the total returns in percentage one could have expected over this period."
    
    return [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [7]:
def analyze_stock_trends(stock_symbol):
    stock_data_page = YahooFinanceWebsite(stock_symbol)
    title, stock_table_data = stock_data_page.get_stock_data()
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = build_stock_analysis_prompt(stock_symbol, title, stock_table_data)
    )
    return response.choices[0].message.content

def display_analysis(stock_symbol):
    display(Markdown(analyze_stock_trends(stock_symbol)))

In [8]:
display_analysis("GOOG")

# Analysis of GOOG Stock Trends over the Past Year

## Overview
The dataset presents the weekly performance of GOOG's stock prices over a year, detailing key metrics such as opening, closing, highest, and lowest prices, alongside volume.

---

## Summary of Key Trends

1. **Price Movement**:
   - The opening price saw an increase from around **143.43** on January 15, 2024, to **191.34** at the end of the timeline on January 13, 2025.
   - The closing price at the end shows **191.77**, indicating a slight appreciation compared to the opening price one year back.

2. **Fluctuations**:
   - The highest price recorded during this period reached **202.14** on January 6, 2025, while the lowest was **137.07** on March 11, 2024. This indicates relatively high volatility, where prices fluctuated significantly throughout the year.

3. **Overall Growth**:
   - The lowest recorded price during this period also provides context. Initially at **143.43**, the stock has seen a **substantial increase**, with the price ending above **191**, suggesting overall bullish sentiment in the stock market.

4. **Volume Dynamics**:
   - Volume ranged greatly, with peaks over **156 million** shares traded on December 16, 2024, indicating high investor interest during certain weeks. Such volume dynamics may relate to earnings announcements or market news impacting the stock price.

5. **Dividend Information**:
   - The data indicates periodic dividends marked with **0.20**, suggesting that investors received income during this period, which can be attractive for long-term investors.

---

## Total Returns Calculation

To calculate the total return percentage on GOOG's stock over this period, follow the formula:

```plaintext
Total Return (%) = ((Ending Price - Beginning Price) / Beginning Price) * 100
```

- **Beginning Price (Jan 15, 2024)**: 147.97 (Closing price)
- **Ending Price (Jan 13, 2025)**: 191.77 (Closing price)

### Calculation:
```plaintext
Total Return (%) = ((191.77 - 147.97) / 147.97) * 100
Total Return (%) = (43.80 / 147.97) * 100 = 29.6%
```

---

## Conclusion
Over the past year, GOOG has shown a **positive growth trend**, with total returns of around **29.6%** indicating solid performance compared to its starting point. The stock experienced significant price movements along the way, reflecting active trading and possibly influenced by market conditions associated with economic factors or company-specific events.

Investors looking to engage in this stock may find its history of rising prices and dividends appealing, while keeping an eye on the price fluctuations to align with market conditions for optimal trading strategies.

In [ ]:
display_analysis("PFE")

In [ ]:
display_analysis("AAPL")

In [9]:
display_analysis("INTC")

## Analyze and Summarize Trends for INTC Stock

### Overview
The stock data for Intel Corporation (INTC) covers weekly metrics such as opening price, closing price, high, low, adjusted closing price, and trading volume over a year. The final rows of the dataset lead up to the week of January 13, 2025.

### Key Trend Observations

1. **Price Fluctuations**:
   - The stock has seen significant volatility over the analyzed period, with prices ranging from a low of **18.89** to a high of **50.30**.
   - Early in **January 2024**, the stock was at **48.85** and experienced a drop to around **19.11** by **January 2025**.

2. **Peak and Decline**:
   - There was a notable peak around **January 22, 2024**, where the price reached **48.85**. 
   - After this peak, a downward trend followed, leading to successive weekly declines, suggesting a bear market.

3. **Recent Performance**:
   - The most recent closing price on **January 13, 2025**, was **19.11**, showing a significant decrease from earlier highs.
   - The average closing price during the preceding months shows a downward trend suggesting profit-taking or negative market sentiment.

4. **Volume**:
   - Trading volumes showed considerable variation, with some weeks achieving as much as **639,840,000** trades, indicating high activity during significant price moves. 
   - The volume has been particularly volatile, which can indicate strong speculation or response to market news.

### Total Returns Calculation

To calculate the total returns in percentage over the period:

1. **Initial Price** on **January 15, 2024**: **48.15**
2. **Final Price** on **January 13, 2025**: **19.11**

The formula for total return is:

\[
\text{Total Return (\%)} = \left( \frac{\text{Final Price} - \text{Initial Price}}{\text{Initial Price}} \right) \times 100
\]

Substituting the values:

\[
\text{Total Return (\%)} = \left( \frac{19.11 - 48.15}{48.15} \right) \times 100
\]

\[
\text{Total Return (\%)} = \left( \frac{-29.04}{48.15} \right) \times 100 \approx -60.4\%
\]

### Conclusion
Investment in Intel (INTC) over this period would have resulted in a total return of **approximately -60.4%**, reflecting a significant decline in stock value. New investors should carefully consider this trend along with market conditions when evaluating potential investments in INTC.